In [1]:
import torch
import unsloth
import random
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import standardize_sharegpt
from unsloth.chat_templates import train_on_responses_only
from unsloth import FastLanguageModel, is_bfloat16_supported  
from transformers import TrainingArguments, TextStreamer, DataCollatorForSeq2Seq
from trl import SFTTrainer 
from datasets import load_dataset, concatenate_datasets

from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel, PeftConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[2025-03-25 14:58:08,379] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="SFT3",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    token=""
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.2",
)


==((====))==  Unsloth 2025.3.14: Fast Llama patching. Transformers: 4.48.1.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0a0+df5bbc09d1.nv24.11. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model.load_adapter("SFT4-1-adapter", token = '')
adapter1_state_dict = model.state_dict()

In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="SFT3",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    token=""
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.2",
)

==((====))==  Unsloth 2025.3.14: Fast Llama patching. Transformers: 4.48.1.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0a0+df5bbc09d1.nv24.11. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model.load_adapter("SFT4-2-adapter", token = '')
adapter2_state_dict = model.state_dict()

In [28]:
merged_state_dict = {}
weight1, weight2 = 0.4, 0.6

for key in adapter1_state_dict.keys():
    if key in adapter2_state_dict:
        merged_state_dict[key] = (
            weight1 * adapter1_state_dict[key] + 
            weight2 * adapter2_state_dict[key]
        )

In [31]:
model = FastLanguageModel.get_peft_model(
    model,
    r=12,
    lora_alpha=24,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "gate_proj", "o_proj"],  # 将两个配置合并
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [32]:
model.load_state_dict(merged_state_dict, strict=False)

_IncompatibleKeys(missing_keys=['base_model.model.model.embed_tokens.weight', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.weight', 'base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.base_layer.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 

In [ ]:
model_location = "SFT5"
tokenizer.push_to_hub(model_location, token='')
model.push_to_hub_merged(model_location, tokenizer, save_method='merged_16bit', token='')

Unsloth: You are pushing to hub, but you passed your HF username = EiLaTe.
We shall truncate EiLaTe/LLaMA3.2-3B-STAT-SFT5 to LLaMA3.2-3B-STAT-SFT5


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 504.48 out of 754.3 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 80.25it/s]


Unsloth: Saving tokenizer... Done.
Done.
Saved merged model to https://huggingface.co/EiLaTe/LLaMA3.2-3B-STAT-SFT5
